<a href="https://colab.research.google.com/github/mastertpf/3d/blob/main/Induktionsmotor_momentkurver_UI_realistisk_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Realistiske momentkurver for 3-faset kortslutningsmotor

Denne version har fjernet valgene for *konstant effekt* og de fire importknapper.

**Workflow:**
1. Indtast effekt, fuldlast-omdrejningstal, poltal og frekvens.
2. Vælg NEMA-design og belastningstype.
3. Tryk *Plot*.

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import Dropdown, FloatText, IntText, Button, HBox, VBox, Output, Layout, Accordion, Checkbox
from IPython.display import display, clear_output
from math import pi, sqrt

plt.rcParams['figure.figsize'] = (8,5)
plt.rcParams['axes.grid'] = True

def synchronous_speed(poles:int, freq_hz:float) -> float:
    return 120.0 * freq_hz / max(2, poles)

def rated_torque(P_kW, n_fl):
    return (P_kW*1000) / (2*pi*n_fl/60)

def nema_templates(design):
    t = {
        'A': dict(R1=0.20,X1=0.45,R2p=0.18,X2p=0.45,Xm=10.0,V_ll=400.0),
        'B': dict(R1=0.25,X1=0.55,R2p=0.22,X2p=0.55,Xm=9.0,V_ll=400.0),
        'C': dict(R1=0.30,X1=0.60,R2p=0.35,X2p=0.60,Xm=8.0,V_ll=400.0),
        'D': dict(R1=0.35,X1=0.65,R2p=0.80,X2p=0.65,Xm=7.0,V_ll=400.0)
    }
    return t[design]

def thevenin_equivalents(R1,X1,Xm,V_phase):
    Z1=complex(R1,X1); Zm=complex(0,Xm)
    Vth=V_phase*(Zm/(Z1+Zm)); Zth=Z1*Zm/(Z1+Zm)
    return abs(Vth), Zth.real, Zth.imag

def induction_torque_curve(R1,X1,R2p,X2p,Xm,V_ll,poles,freq,n_points=800):
    n_sync=synchronous_speed(poles,freq)
    w_sync=2*pi*n_sync/60
    Vp=V_ll/sqrt(3)
    Vth,Rth,Xth=thevenin_equivalents(R1,X1,Xm,Vp)
    s=np.geomspace(1e-3,1.0,n_points)[::-1]
    denom=(Rth+(R2p/np.clip(s,1e-6,1)))**2+(Xth+X2p)**2
    T=(3*(Vth**2)*(R2p/np.clip(s,1e-6,1)))/(w_sync*denom)
    n=(1-s)*n_sync
    return n,T,n_sync

def scale_to_full_load(n,T,n_fl,T_fl):
    i=np.argmin(np.abs(n-n_fl)); return T*(T_fl/max(T[i],1e-9))

def load_torque_curve(n,n_fl,T_fl,typ):
    x=np.clip(n/n_fl,1e-3,5)
    if typ in ('Pumpe/Ventilator','Ventilator/Fan','Centrifugal pumpe'): return T_fl*(x**2)
    elif typ in ('Transportbånd','Mixer/Extruder','Konstant moment'): return np.full_like(n,T_fl)
    elif typ in ('Stenknuser','Crusher (høj startmoment)'):
        return T_fl*(1+0.8*(1-x))
    return np.full_like(n,T_fl)

P=FloatText(value=75,description='Effekt [kW]:',layout=Layout(width='200px'))
nfl=FloatText(value=1485,description='Fuldlast rpm:',layout=Layout(width='200px'))
poles=IntText(value=4,description='Poler:',layout=Layout(width='120px'))
freq=FloatText(value=50,description='Frekvens [Hz]:',layout=Layout(width='160px'))
design=Dropdown(options=['A','B','C','D'],value='B',description='Design:')
load=Dropdown(options=['Pumpe/Ventilator','Transportbånd','Stenknuser','Mixer/Extruder','Ventilator/Fan','Centrifugal pumpe','Konstant moment'],value='Pumpe/Ventilator',description='Belastning:')
tmpl=nema_templates(design.value)
R1=FloatText(value=tmpl['R1'],description='R1 [Ω]:'); X1=FloatText(value=tmpl['X1'],description='X1 [Ω]:'); R2=FloatText(value=tmpl['R2p'],description="R2' [Ω]:"); X2=FloatText(value=tmpl['X2p'],description="X2' [Ω]:"); Xm=FloatText(value=tmpl['Xm'],description='Xm [Ω]:'); Vll=FloatText(value=tmpl['V_ll'],description='V_ll [V]:'); link=Checkbox(value=True,description='Skaler til T_FL')
acc=Accordion(children=[HBox([R1,X1,R2,X2,Xm,Vll,link])]); acc.set_title(0,'Avanceret')
btn=Button(description='Plot',button_style='primary'); out=Output()
def upd(c):
 t=nema_templates(c['new']); R1.value=t['R1'];X1.value=t['X1'];R2.value=t['R2p'];X2.value=t['X2p'];Xm.value=t['Xm'];Vll.value=t['V_ll']
design.observe(upd,names='value')
def run(_):
 with out:
  clear_output(wait=True)
  pol = poles.value # Correctly assign poles value
  freq_val = freq.value # Correctly assign freq value
  T_fl=rated_torque(P.value,nfl.value)
  n,T,n_sync=induction_torque_curve(R1.value,X1.value,R2.value,X2.value,Xm.value,Vll.value,pol,freq_val)
  if link.value: T=scale_to_full_load(n,T,nfl.value,T_fl)
  Tl=load_torque_curve(n,nfl.value,T_fl,load.value)
  plt.plot(n,T,label=f'Motor {design.value}'); plt.plot(n,Tl,label=f'Belastning: {load.value}')
  plt.axvline(nfl.value,ls='--',lw=1); plt.scatter([nfl.value],[T_fl]); plt.text(nfl.value,T_fl,'  FL',va='bottom')
  plt.xlabel('rpm');plt.ylabel('Moment [Nm]');plt.legend();plt.tight_layout();plt.show()
  print(f'n_sync={synchronous_speed(pol,freq_val):.1f} rpm, T_FL={T_fl:.1f} Nm')
btn.on_click(run)
display(VBox([HBox([P,nfl,poles,freq,design]),HBox([load,btn]),acc,out]))